In [0]:
from pyspark.sql.functions import col, round

In [0]:
tenant_id = "your_tenant_id"
client_id = "your_client_id"
client_secret = "your_app_secret"

storage_account = "olympicdata7"    
container = "tokyo-olympic-data"        

# OAuth config for this storage account
spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(
  f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net",
  "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", client_secret)

# Token endpoint (v1 works broadly)
spark.conf.set(
  f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
  f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
)

# CSV raw files' path
files_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/raw-data"


In [0]:
athletes_df = spark.read.format("csv").option("header", "true").option("inferSchema","true").load(f"{files_path}/Athletes.csv")
coaches_df = spark.read.format("csv").option("header", "true").option("inferSchema","true").load(f"{files_path}/Coaches.csv")
entriesgender_df = spark.read.format("csv").option("header", "true").option("inferSchema","true").load(f"{files_path}/EntriesGender.csv")
medals_df = spark.read.format("csv").option("header", "true").option("inferSchema","true").load(f"{files_path}/Medals.csv")
teams_df = spark.read.format("csv").option("header", "true").option("inferSchema","true").load(f"{files_path}/Teams.csv")

In [0]:
display(entriesgender_df)

Discipline,Female,Male,Total
3x3 Basketball,32,32,64
Archery,64,64,128
Artistic Gymnastics,98,98,196
Artistic Swimming,105,0,105
Athletics,969,1072,2041
Badminton,86,87,173
Baseball/Softball,90,144,234
Basketball,144,144,288
Beach Volleyball,48,48,96
Boxing,102,187,289


In [0]:
entriesgender_df.printSchema()

root
 |-- Discipline: string (nullable = true)
 |-- Female: string (nullable = true)
 |-- Male: string (nullable = true)
 |-- Total: string (nullable = true)



In [0]:
medals_df.show()

+----+--------------------+----+------+------+-----+-------------+
|Rank|            Team/NOC|Gold|Silver|Bronze|Total|Rank by Total|
+----+--------------------+----+------+------+-----+-------------+
|   1|United States of ...|  39|    41|    33|  113|            1|
|   2|People's Republic...|  38|    32|    18|   88|            2|
|   3|               Japan|  27|    14|    17|   58|            5|
|   4|       Great Britain|  22|    21|    22|   65|            4|
|   5|                 ROC|  20|    28|    23|   71|            3|
|   6|           Australia|  17|     7|    22|   46|            6|
|   7|         Netherlands|  10|    12|    14|   36|            9|
|   8|              France|  10|    12|    11|   33|           10|
|   9|             Germany|  10|    11|    16|   37|            8|
|  10|               Italy|  10|    10|    20|   40|            7|
|  11|              Canada|   7|     6|    11|   24|           11|
|  12|              Brazil|   7|     6|     8|   21|          

In [0]:
medals_df = (medals_df
.withColumnRenamed("Team/NOC","NOC")
.withColumnRenamed("Rank by Total","Rank_by_Total")
)

In [0]:
# Create weighted point column 
medals_df = medals_df.withColumn(
    "weighted_points_ioc",
    col("Gold")*3 + col("Silver")*2 + col("Bronze")
)
medals_df.show(10)


+----+--------------------+----+------+------+-----+-------------+-------------------+
|Rank|                 NOC|Gold|Silver|Bronze|Total|Rank_by_Total|weighted_points_ioc|
+----+--------------------+----+------+------+-----+-------------+-------------------+
|   1|United States of ...|  39|    41|    33|  113|            1|                232|
|   2|People's Republic...|  38|    32|    18|   88|            2|                196|
|   3|               Japan|  27|    14|    17|   58|            5|                126|
|   4|       Great Britain|  22|    21|    22|   65|            4|                130|
|   5|                 ROC|  20|    28|    23|   71|            3|                139|
|   6|           Australia|  17|     7|    22|   46|            6|                 87|
|   7|         Netherlands|  10|    12|    14|   36|            9|                 68|
|   8|              France|  10|    12|    11|   33|           10|                 65|
|   9|             Germany|  10|    11|    

In [0]:
medals_df.printSchema()

root
 |-- Rank: integer (nullable = true)
 |-- NOC: string (nullable = true)
 |-- Gold: integer (nullable = true)
 |-- Silver: integer (nullable = true)
 |-- Bronze: integer (nullable = true)
 |-- Total: integer (nullable = true)
 |-- Rank_by_Total: integer (nullable = true)



In [0]:
medals_df.show(5)

+----+--------------------+----+------+------+-----+-------------+
|Rank|                 NOC|Gold|Silver|Bronze|Total|Rank_by_Total|
+----+--------------------+----+------+------+-----+-------------+
|   1|United States of ...|  39|    41|    33|  113|            1|
|   2|People's Republic...|  38|    32|    18|   88|            2|
|   3|               Japan|  27|    14|    17|   58|            5|
|   4|       Great Britain|  22|    21|    22|   65|            4|
|   5|                 ROC|  20|    28|    23|   71|            3|
+----+--------------------+----+------+------+-----+-------------+
only showing top 5 rows


In [0]:
# Find the top countries with the most number of medals
top_medal_countries = medals_df.orderBy("Total", ascending=False).show()

+----+--------------------+----+------+------+-----+-------------+
|Rank|            Team/NOC|Gold|Silver|Bronze|Total|Rank by Total|
+----+--------------------+----+------+------+-----+-------------+
|   1|United States of ...|  39|    41|    33|  113|            1|
|   2|People's Republic...|  38|    32|    18|   88|            2|
|   5|                 ROC|  20|    28|    23|   71|            3|
|   4|       Great Britain|  22|    21|    22|   65|            4|
|   3|               Japan|  27|    14|    17|   58|            5|
|   6|           Australia|  17|     7|    22|   46|            6|
|  10|               Italy|  10|    10|    20|   40|            7|
|   9|             Germany|  10|    11|    16|   37|            8|
|   7|         Netherlands|  10|    12|    14|   36|            9|
|   8|              France|  10|    12|    11|   33|           10|
|  11|              Canada|   7|     6|    11|   24|           11|
|  12|              Brazil|   7|     6|     8|   21|          

In [0]:
# Calculate the average # of entries by gender for each discipline 
avg_entries_by_gender = entriesgender_df.withColumn(
    'Avg_Female',round(col('Female') / col('Total'),2)
).withColumn(
    'Avg_Male',round(col('Male') / col('Total'),2)
)
avg_entries_by_gender.show()

+--------------------+------+----+-----+----------+--------+
|          Discipline|Female|Male|Total|Avg_Female|Avg_Male|
+--------------------+------+----+-----+----------+--------+
|      3x3 Basketball|    32|  32|   64|       0.5|     0.5|
|             Archery|    64|  64|  128|       0.5|     0.5|
| Artistic Gymnastics|    98|  98|  196|       0.5|     0.5|
|   Artistic Swimming|   105|   0|  105|       1.0|     0.0|
|           Athletics|   969|1072| 2041|      0.47|    0.53|
|           Badminton|    86|  87|  173|       0.5|     0.5|
|   Baseball/Softball|    90| 144|  234|      0.38|    0.62|
|          Basketball|   144| 144|  288|       0.5|     0.5|
|    Beach Volleyball|    48|  48|   96|       0.5|     0.5|
|              Boxing|   102| 187|  289|      0.35|    0.65|
|        Canoe Slalom|    41|  41|   82|       0.5|     0.5|
|        Canoe Sprint|   123| 126|  249|      0.49|    0.51|
|Cycling BMX Frees...|    10|   9|   19|      0.53|    0.47|
|  Cycling BMX Racing|  

In [0]:
# CSV transformed files' path
transformed_files_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/transformed-data-csv/"

In [0]:
# For Azure Synapse usage 

athletes_df.write.mode("overwrite").option("header","true").parquet(f"{transformed_files_path}/athletes.parquet")
coaches_df.write.mode("overwrite").option("header","true").parquet(f"{transformed_files_path}/coaches.parquet")
entriesgender_df.write.mode("overwrite").option("header","true").parquet(f"{transformed_files_path}/entriesgender.parquet")
medals_df.write.mode("overwrite").option("header","true").parquet(f"{transformed_files_path}/medals.parquet")
teams_df.write.mode("overwrite").option("header","true").parquet(f"{transformed_files_path}/teams.parquet")
avg_entries_by_gender.write.mode("overwrite").option("header","true").parquet(f"{transformed_files_path}/avg_entries_by_gender.parquet")





In [0]:
# For tableau visualization usage
athletes_df.write.mode("overwrite").option("header","true").csv(f"{transformed_files_path}/athletes.csv")
athletes_df.write.mode("overwrite").option("header","true").csv(f"{transformed_files_path}/entriesgender.csv")
medals_df.write.mode("overwrite").option("header","true").csv(f"{transformed_files_path}/medals.csv")
coaches_df.write.mode("overwrite").option("header","true").csv(f"{transformed_files_path}/coaches.csv")
teams_df.write.mode("overwrite").option("header","true").csv(f"{transformed_files_path}/teams.csv")
